# Data Preparation

In [133]:
import pandas as pd

In [134]:
data = pd.read_csv("data/2024ReportCardData.csv", usecols=lambda col: not col.startswith('Unnamed'))

/var/folders/p6/tqnlcwnd075_z06_lxt45pv40000gn/T/ipykernel_26234/802293298.py:1: DtypeWarning: Columns (0: TSI_CSI, 1: TSI_CSI_Detail, 2: Math Proficiency, 3: English Proficiency, 4: US History Proficiency, 5: Science Proficiency, 6: Number of EL Students Proficient, 7: EL Growth, 8: Math Performance Level 1, 9: Math Performance Students Level 1, 10: Math Performance Level 2, 11: Math Performance Students Level 2, 12: Math Performance Level 3, 13: Math Performance Students Level 3, 14: Math Performance Level 4, 15: Math Performance Students Level 4, 16: Math Performance Level 5, 17: Math Performance Students Level 5, 18: English Performance Level 1, 19: English Performance Students Level 1, 20: English Performance Level 2, 21: English Performance Students Level 2, 22: English Performance Level 3, 23: English Performance Students Level 3, 24: English Performance Level 4, 25: English Performance Students Level 4, 26: English Performance Level 5, 27: English Performance Students Level 5, 

# District Analsyis

In [135]:
#sort data by district, students with disabilities, students without disabilities, and graduation rates
districts = data[data["Type"] == "District"]
special_ed_district = districts[districts["Subgroup"].isin(["Students with Disabilities", "Students without Disabilities"])]
se_district_grad = special_ed_district[["District_Name", "Subgroup", "Graduation Rate"]]
se_district_grad
pivot = pd.pivot_table(se_district_grad, values = "Graduation Rate", index = "District_Name", columns = "Subgroup", aggfunc = "sum").reset_index()
pivot = pivot.rename(columns = {"Students with Disabilities": "Students_With_Disabilities_Grad_Rate", "Students without Disabilities": "Students_Without_Disabilities_Grad_Rate"})
pivot.sort_values("Students_With_Disabilities_Grad_Rate")

Subgroup,District_Name,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate
0,Aberdeen School District,0.0,92.1
49,Holly Springs School District,0.0,96.9
55,Jefferson County School District,0.0,92.6
56,Jefferson Davis County School District,0.0,90.8
57,Joel E. Smilow Collegiate,0.0,0.0
...,...,...,...
111,Quitman School District,90.0,96.3
8,Benton County School District,90.0,98.1
32,Enterprise School District,90.9,97.1
38,Greene County School District,100.0,92.3


In [136]:
pivot["Students_With_Disabilities_Grad_Rate"].value_counts()

Students_With_Disabilities_Grad_Rate
0.0     44
75.0     5
54.5     4
50.0     4
47.4     3
        ..
65.9     1
80.8     1
27.3     1
23.1     1
25.8     1
Name: count, Length: 77, dtype: int64

## Takeaway 

44 school districts (approximately 30% of MS school districts) have a "0%" graduation rate for students with disabilities, indicating there either are no students with disabilities or there is missing data. Recommended next steps are to determine why these zero values exist. 

If a school district doesn't have any students with disabilities, it could be an indication students aren't tested and/or diagnosed for disabilities and therefore aren't given the specialized education they need. Could be interesting to see if the entire student body's grades indicate this as a possibility.

In [137]:
#remove zero/missing values for now to identify the school districts that have data available to analyze now
dis_df = pivot[pivot.Students_With_Disabilities_Grad_Rate != 0.0]
dis_df["Percentage Points Difference"] = (dis_df["Students_Without_Disabilities_Grad_Rate"] - dis_df["Students_With_Disabilities_Grad_Rate"])
dis_df["Graduation Ratio"] = (dis_df["Students_Without_Disabilities_Grad_Rate"] / dis_df["Students_With_Disabilities_Grad_Rate"])
dis_df

Subgroup,District_Name,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate,Percentage Points Difference,Graduation Ratio
1,Alcorn School District,65.2,95.8,30.6,1.469325
4,Amory School District,75.0,93.1,18.1,1.241333
7,Bay St Louis Waveland School District,75.0,96.7,21.7,1.289333
8,Benton County School District,90.0,98.1,8.1,1.090000
9,Biloxi Public School District,77.5,92.6,15.1,1.194839
...,...,...,...,...,...
139,West Jasper Consolidated Schools,80.8,97.2,16.4,1.202970
140,West Point Consolidated School District,30.0,82.6,52.6,2.753333
142,Western Line School District,27.3,78.5,51.2,2.875458
143,Wilkinson County School District,23.1,79.2,56.1,3.428571


In [138]:
dis_df.describe()

Subgroup,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate,Percentage Points Difference,Graduation Ratio
count,103.000000,103.000000,103.000000,103.000000
mean,62.087379,91.125243,29.037864,1.608604
std,17.304871,5.567971,14.537141,0.570871
min,18.200000,75.600000,-7.700000,0.923000
25%,50.000000,87.800000,17.950000,1.239853
50%,64.100000,92.500000,28.300000,1.447900
75%,74.550000,95.600000,39.850000,1.760589
max,100.000000,98.900000,66.000000,4.626374


In [139]:
worst_grad_rates = dis_df.sort_values("Students_With_Disabilities_Grad_Rate")
worst_grad_rates.head(20)

Subgroup,District_Name,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate,Percentage Points Difference,Graduation Ratio
122,South Pike School District,18.2,84.2,66.0,4.626374
143,Wilkinson County School District,23.1,79.2,56.1,3.428571
145,Yazoo City Municipal School District,25.8,76.9,51.1,2.980620
142,Western Line School District,27.3,78.5,51.2,2.875458
19,Clarksdale Municipal School District,28.6,85.3,56.7,2.982517
140,West Point Consolidated School District,30.0,82.6,52.6,2.753333
46,Hazlehurst City School District,30.0,83.3,53.3,2.776667
94,Noxubee County School District,35.7,88.2,52.5,2.470588
91,North Panola School District,35.7,85.7,50.0,2.400560
23,Coffeeville School District,36.4,89.7,53.3,2.464286


In [140]:
top_20_diff = dis_df.sort_values("Percentage Points Difference", ascending = False).head(20)
top_20_diff

Subgroup,District_Name,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate,Percentage Points Difference,Graduation Ratio
122,South Pike School District,18.2,84.2,66.0,4.626374
136,Wayne County School District,38.5,96.3,57.8,2.501299
19,Clarksdale Municipal School District,28.6,85.3,56.7,2.982517
143,Wilkinson County School District,23.1,79.2,56.1,3.428571
23,Coffeeville School District,36.4,89.7,53.3,2.464286
46,Hazlehurst City School District,30.0,83.3,53.3,2.776667
140,West Point Consolidated School District,30.0,82.6,52.6,2.753333
94,Noxubee County School District,35.7,88.2,52.5,2.470588
84,Natchez-Adams School District,41.4,93.4,52.0,2.256039
142,Western Line School District,27.3,78.5,51.2,2.875458


In [141]:
top_20_rate = dis_df.sort_values("Graduation Ratio", ascending = False).head(20)
top_20_rate

Subgroup,District_Name,Students_With_Disabilities_Grad_Rate,Students_Without_Disabilities_Grad_Rate,Percentage Points Difference,Graduation Ratio
122,South Pike School District,18.2,84.2,66.0,4.626374
143,Wilkinson County School District,23.1,79.2,56.1,3.428571
19,Clarksdale Municipal School District,28.6,85.3,56.7,2.982517
145,Yazoo City Municipal School District,25.8,76.9,51.1,2.980620
142,Western Line School District,27.3,78.5,51.2,2.875458
46,Hazlehurst City School District,30.0,83.3,53.3,2.776667
140,West Point Consolidated School District,30.0,82.6,52.6,2.753333
136,Wayne County School District,38.5,96.3,57.8,2.501299
94,Noxubee County School District,35.7,88.2,52.5,2.470588
23,Coffeeville School District,36.4,89.7,53.3,2.464286


Students without disabilities are at least twice as likely to graduate compared to students with disabilities at 15 school districts in Mississippi 

Students with disabilities have lower graduation rates than the state average at 46 school districts (appx 30%)

In [142]:
top_3_diff = dis_df.sort_values("Percentage Points Difference", ascending = False).head(20)

In [143]:
top_20 = top_20_diff["District_Name"].unique()
top_3 = top_3_diff["District_Name"].unique()

In [144]:
dropout = pd.read_csv("data/ms_dropout.csv")
dropout = dropout.rename(columns={"District": "District_Name"})
dropout["District_Name"] = (
    dropout["District_Name"]
    .str.replace("\xa0", "", regex=False)
    .str.strip()
)
top_20_drop = dropout[dropout["District_Name"].isin(top_20)]
top_20_drop

,District_Name,N-Count,4-Year Graduation Rate,Dropout Rate
14,Chickasaw County School District,174,92.5%,6.3%
17,Clarksdale Municipal School District,153,86.3%,11.8%
18,Cleveland School District,207,83.6%,14.0%
21,Coffeeville School District,34,73.5%,14.7%
25,Corinth School District,156,89.7%,7.7%
31,Forest Municipal School District,124,68.5%,25.8%
40,Gulfport School District,441,90.2%,8.2%
44,Hazlehurst City School District,115,70.4%,19.1%
45,Hinds County School District,433,88.5%,9.2%
81,Natchez-Adams School District,187,89.8%,5.9%


All districts but two had higher dropout rates in the 2025-2026 school year than the Mississippi dropout average (appx 7%)

# Mississippi Department of Education District Grading System

In [145]:
top_20_districts = special_ed_district[special_ed_district["District_Name"].isin(top_20)]
top_20_district_grades = top_20_districts[["School_Name", "Grade"]].copy()
top_20_district_grades = top_20_district_grades.drop_duplicates()
grade_dist = top_20_district_grades["Grade"].value_counts().reset_index()
grade_dist["%"] = (grade_dist["count"] / grade_dist["count"].sum())*100
grade_dist = grade_dist.sort_values("Grade")
grade_dist

,Grade,count,%
4,A,1,5.0
1,B,6,30.0
0,C,8,40.0
3,D,2,10.0
2,F,3,15.0


In [146]:
import altair as alt
alt.Chart(grade_dist).mark_bar().encode(
    x = "count",
    y = "Grade"
).properties(
    title = "District Grade Distribution of Top 20 School Districts with Disability Grad Gap"
)

alt.Chart(...)

In [147]:
all_districts = data[data["Type"] == "District"]
all_districts = all_districts[all_districts["Subgroup"] == "All"]
all_grade_dist = all_districts["Grade"].value_counts().reset_index()
all_grade_dist["%"] = (all_grade_dist["count"] / all_grade_dist["count"].sum())*100
all_grade_dist

,Grade,count,%
0,A,52,35.374150
1,B,42,28.571429
2,C,36,24.489796
3,D,11,7.482993
4,F,6,4.081633


In [148]:
alt.Chart(all_grade_dist).mark_bar().encode(
    x = "count",
    y = "Grade",
).properties(
    title = "District Grade Distribution of All Mississippi School Districts"
)

alt.Chart(...)

The school districts with the widest graduation gap between disabled and non-disabled students (top 20) received low "grades" from the Mississippi Department of Education. The Mississippi Statewide Accountability System assigns a performance rating of A, B, C, D, or F to each school and district based on points earned through student achievement, individual growth, and participation on statewide assessments; and for graduation, college and career readiness, and participation and performance on accelerated coursework for high schools. 

# Student Proficiency Analysis

In [149]:
se_district_grad_prof = special_ed_district[["District_Name", "Subgroup", "Graduation Rate", "English Proficiency", "Math Proficiency", "Science Proficiency"]]
se_district_grad_prof
dis_grad_prof = se_district_grad_prof.dropna()
dis_grad_prof["Math Proficiency"] = dis_grad_prof["Math Proficiency"].astype("float64")
dis_grad_prof["English Proficiency"] = dis_grad_prof["English Proficiency"].astype("float64")
dis_grad_prof["Graduation Rate"] = dis_grad_prof["Graduation Rate"].astype("float64")
dis_grad_prof["Science Proficiency"] = dis_grad_prof["Science Proficiency"].astype("float64")

In [150]:
alt.Chart(dis_grad_prof).mark_circle(size=60).encode(
    x='Graduation Rate',
    y='English Proficiency',
    color='Subgroup',
    tooltip=['District_Name', 'Graduation Rate', 'English Proficiency']
).interactive()

alt.Chart(...)

In [151]:
alt.Chart(dis_grad_prof).mark_circle(size=60).encode(
    x='Graduation Rate',
    y='Math Proficiency',
    color='Subgroup',
    tooltip=['District_Name', 'Graduation Rate', 'Math Proficiency']
).interactive()

alt.Chart(...)

There appears to be a positive correlation between subject proficiency rates and graduation rate. Both Math and English proficiency for students with disabilities is low. 

However, the section at the center of the chart, where orange and blue converge, is interesting. There are schools where proficiency percentages are similar (on the lower end), yet graduation rates appear to be vastly different. It looks as if students without disabilities are graduating at much higher rates, even when proficiency levels among students with disabilities are comparable. Although, it's important to note that students with and without disabilities have different standards for what "proficiency" means. 

# School Analysis

In [152]:
import numpy as np
dist_schools = data[data["Type"] == "School"]
sped_schools = dist_schools[dist_schools["Subgroup"].isin(["Students with Disabilities", "Students without Disabilities"])]
sped_schools = sped_schools[sped_schools["District_Name"].isin(top_3)]
sped_schools.loc[sped_schools["% Chronic Absenteeism"] == '<5%', "% Chronic Absenteeism"] = 5
sped_schools["% Chronic Absenteeism"] = sped_schools["% Chronic Absenteeism"].astype("float64")
sped_schools_copy = sped_schools[["District_Name", "School_Name", "Subgroup", "Math Proficiency", "English Proficiency", "US History Proficiency", "Science Proficiency"]].copy()
sped_schools_copy.loc[sped_schools_copy["Math Proficiency"] == '<1%', "Math Proficiency"] = 0.9
sped_schools_copy.loc[sped_schools_copy["English Proficiency"] == '<1%', "English Proficiency"] = 0.9
sped_schools_copy.loc[sped_schools_copy["US History Proficiency"] == '<1%', "US History Proficiency"] = 0.9
sped_schools_copy.loc[sped_schools_copy["Science Proficiency"] == '<1%', "Science Proficiency"] = 0.9
sped_schools_copy["Math Proficiency"] = sped_schools_copy["Math Proficiency"].replace(r"^\s*$", np.nan, regex = True)
sped_schools_copy["English Proficiency"] = sped_schools_copy["English Proficiency"].replace(r"^\s*$", np.nan, regex = True)
sped_schools_copy["US History Proficiency"] = sped_schools_copy["US History Proficiency"].replace(r"^\s*$", np.nan, regex = True)
sped_schools_copy["Science Proficiency"] = sped_schools_copy["Science Proficiency"].replace(r"^\s*$", np.nan, regex = True)
sped_schools_copy["Math Proficiency"] = sped_schools_copy["Math Proficiency"].astype("float64")
sped_schools_copy["English Proficiency"] = sped_schools_copy["English Proficiency"].astype("float64")
sped_schools_copy["US History Proficiency"] = sped_schools_copy["US History Proficiency"].astype("float64")
sped_schools_copy["Science Proficiency"] = sped_schools_copy["Science Proficiency"].astype("float64")
sped_schools_copy

,District_Name,School_Name,Subgroup,Math Proficiency,English Proficiency,US History Proficiency,Science Proficiency
108,Natchez-Adams School District,Mc Laurin Elementary School,Students with Disabilities,22.4,16.9,NaN,NaN
109,Natchez-Adams School District,Mc Laurin Elementary School,Students without Disabilities,45.7,37.5,NaN,NaN
126,Natchez-Adams School District,Morgantown Elementary,Students with Disabilities,6.8,13.6,NaN,18.2
127,Natchez-Adams School District,Morgantown Elementary,Students without Disabilities,19.9,48.6,NaN,51.3
142,Natchez-Adams School District,Natchez Early College@Co-Lin,Students with Disabilities,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
18786,Yazoo City Municipal School District,McCoy Elementary School,Students without Disabilities,27.0,19.7,NaN,33.7
18802,Yazoo City Municipal School District,Bettie E Woolfolk Middle School,Students with Disabilities,8.0,5.7,NaN,8.1
18803,Yazoo City Municipal School District,Bettie E Woolfolk Middle School,Students without Disabilities,18.8,14.8,NaN,20.0
18819,Yazoo City Municipal School District,Yazoo City High School,Students with Disabilities,4.3,0.9,NaN,4.0


In [153]:
south_pike = sped_schools_copy[sped_schools_copy["District_Name"] == "South Pike School District"]
alt.Chart(south_pike).mark_bar().encode(
    x = "School_Name:O",
    y = "Math Proficiency:Q",
    xOffset= "Subgroup:N",
    color = "Subgroup:N"
)


alt.Chart(...)

In [154]:
alt.Chart(south_pike).mark_bar().encode(
    x = "School_Name:O",
    y = "English Proficiency:Q",
    xOffset= "Subgroup:N",
    color = "Subgroup:N"
)

alt.Chart(...)

# Historical Analysis

In [155]:
import glob as glob
root_directory = ('data/')
all_files = glob.glob('data/*.csv')
li = []
for filename in all_files:
    frame = pd.read_csv(filename, index_col = None, header = 0, encoding="UTF-8", dtype = object)
    li.append(frame)
df = pd.concat(li, axis = 0, ignore_index = True)               

In [156]:
working_df = df
working_df.loc[working_df["Math Proficiency"] == '<1%', "Math Proficiency"] = 0.9
working_df.loc[working_df["English Proficiency"] == '<1%', "English Proficiency"] = 0.9
working_df.loc[working_df["US History Proficiency"] == '<1%', "US History Proficiency"] = 0.9
working_df.loc[working_df["Science Proficiency"] == '<1%', "Science Proficiency"] = 0.9
ms = working_df[working_df["School_Name"] == "Mississippi"]
ms = ms[ms["Subgroup"].isin(["Students with Disabilities", "Students without Disabilities"])]
ms["Graduation Rate"] = ms["Graduation Rate"].astype(float)
ms_grad_hist = ms[["School_Year", "Subgroup", "Graduation Rate", "% Chronic Absenteeism"]].copy()
ms_grad_hist

,School_Year,Subgroup,Graduation Rate,% Chronic Absenteeism
22,2018,Students with Disabilities,38.4,17.4
23,2018,Students without Disabilities,89.1,12.4
20657,2019,Students with Disabilities,42.2,
20658,2019,Students without Disabilities,90.1,
27866,2023,Students with Disabilities,70.0,27.3
27867,2023,Students without Disabilities,91.9,23.3
46966,2017,Students with Disabilities,36.4,21.8
46967,2017,Students without Disabilities,88.1,16.1
66543,2022,Students with Disabilities,67.1,27.5
66544,2022,Students without Disabilities,91.5,23.2


In [157]:
alt.Chart(ms_grad_hist).mark_line().encode(
    x="School_Year",
    y = "Graduation Rate",
    color = "Subgroup"
)

alt.Chart(...)

The graduation rate for students without disabilities in Mississippi has remained steady, yet high, while the graduation rate for students with disabilities has increased over the past 7 years, with a slight downturn last school year in 2024. So, the disability gap has been closing, yet remains high for some school districts.

In [158]:
working_df_copy = working_df[["School_Year", "District_Name", "School_Name", "Subgroup", "Type", "Math Proficiency", "English Proficiency", "US History Proficiency", "Science Proficiency"]].copy()
working_df_filtered = working_df_copy[working_df_copy["Subgroup"].isin(["Students with Disabilities", "Students without Disabilities"])]
working_df_filtered

,School_Year,District_Name,School_Name,Subgroup,Type,Math Proficiency,English Proficiency,US History Proficiency,Science Proficiency
22,2018,Mississippi,Mississippi,Students with Disabilities,State,20.4,17.6,20.2,26.7
23,2018,Mississippi,Mississippi,Students without Disabilities,State,51.1,45.5,58.9,60.4
91,2018,Natchez-Adams School District,Natchez-Adams School District,Students with Disabilities,District,19,16.1,18.2,22.5
92,2018,Natchez-Adams School District,Natchez-Adams School District,Students without Disabilities,District,26.7,26.4,34.4,35
113,2018,Natchez-Adams School District,Mc Laurin Elementary School,Students with Disabilities,School,22.7,20.5,,42.9
...,...,...,...,...,...,...,...,...,...
142420,2020,Yazoo City Municipal School District,Webster Street Elementary School,Students without Disabilities,School,0.8,5.2,,
142436,2020,Yazoo City Municipal School District,Bettie E Woolfolk Middle School,Students with Disabilities,School,3.8,2,,
142437,2020,Yazoo City Municipal School District,Bettie E Woolfolk Middle School,Students without Disabilities,School,2,6.2,,7.9
142449,2020,Yazoo City Municipal School District,Yazoo City High School,Students with Disabilities,School,0.9,0.9,,


In [159]:
mississippi = working_df_filtered[working_df_filtered["Type"] == "State"]
mississippi["Math Proficiency"] = mississippi["Math Proficiency"].replace(r"^\s*$", np.nan, regex = True)
mississippi["English Proficiency"] = mississippi["English Proficiency"].replace(r"^\s*$", np.nan, regex = True)
mississippi["US History Proficiency"] = mississippi["US History Proficiency"].replace(r"^\s*$", np.nan, regex = True)
mississippi["Science Proficiency"] = mississippi["Science Proficiency"].replace(r"^\s*$", np.nan, regex = True)
mississippi["Math Proficiency"] = mississippi["Math Proficiency"].astype("float64")
mississippi["English Proficiency"] = mississippi["English Proficiency"].astype("float64")
mississippi["US History Proficiency"] = mississippi["US History Proficiency"].astype("float64")
mississippi["Science Proficiency"] = mississippi["Science Proficiency"].astype("float64")
ms_graph = mississippi[["School_Year", "Subgroup", "Math Proficiency", "English Proficiency", "US History Proficiency", "Science Proficiency"]].copy()
ms_graph

,School_Year,Subgroup,Math Proficiency,English Proficiency,US History Proficiency,Science Proficiency
22,2018,Students with Disabilities,20.4,17.6,20.2,26.7
23,2018,Students without Disabilities,51.1,45.5,58.9,60.4
20657,2019,Students with Disabilities,NaN,NaN,NaN,NaN
20658,2019,Students without Disabilities,NaN,NaN,NaN,NaN
27866,2023,Students with Disabilities,27.6,21.4,39.2,35.8
27867,2023,Students without Disabilities,61.5,53.1,74.5,68.1
46966,2017,Students with Disabilities,17.6,15.8,19.2,30.7
46967,2017,Students without Disabilities,47.0,43.4,56.9,69.2
66543,2022,Students with Disabilities,24.8,19.8,37.4,30.9
66544,2022,Students without Disabilities,56.8,51.5,74.6,63.0


In [160]:
alt.Chart(ms_graph).mark_line().encode(
    x = "School_Year", 
    y = "Math Proficiency",
    color = 'Subgroup'
)

alt.Chart(...)

In [161]:
import altair as alt
alt.Chart(ms_graph).mark_line().encode(
    x = "School_Year", 
    y = "English Proficiency",
    color = 'Subgroup'
)

alt.Chart(...)

In [162]:
alt.Chart(ms_graph).mark_line().encode(
    x = "School_Year", 
    y = "English Proficiency",
    color = 'Subgroup'
)

alt.Chart(...)

In [163]:
alt.Chart(ms_graph).mark_line().encode(
    x = "School_Year", 
    y = "US History Proficiency",
    color = 'Subgroup'
)

alt.Chart(...)

In [164]:
alt.Chart(ms_graph).mark_line().encode(
    x = "School_Year", 
    y = "Science Proficiency",
    color = 'Subgroup'
)

alt.Chart(...)

The graduation rates for students with disabilities seems to follow a similar pattern as the graduation rates for non-disabled students. So, while subject proficiency is increasing, the gap between proficiency levels among disabled and non-disabled students is not closing. 

In [165]:
hist_top10 = working_df_filtered[working_df_filtered["Type"] == "District"]
hist_top10 = hist_top10[hist_top10["School_Name"].isin(["South Pike School District",
"Wilkinson County School District",
"Clarksdale Municipal School District",
"Yazoo City Municipal School District",
"Western Line School District",
"Hazlehurst City School District",
"West Point Consolidated School District",
"Wayne County School District",
"Noxubee County School District",
"Coffeeville School District"])]
hist_top10["Math Proficiency"] = hist_top10["Math Proficiency"].replace(r"^\s*$", np.nan, regex = True)
hist_top10["English Proficiency"] = hist_top10["English Proficiency"].replace(r"^\s*$", np.nan, regex = True)
hist_top10["US History Proficiency"] = hist_top10["US History Proficiency"].replace(r"^\s*$", np.nan, regex = True)
hist_top10["Science Proficiency"] = hist_top10["Science Proficiency"].replace(r"^\s*$", np.nan, regex = True)
hist_top10["Math Proficiency"] = hist_top10["Math Proficiency"].astype("float64")
hist_top10["English Proficiency"] = hist_top10["English Proficiency"].astype("float64")
hist_top10["US History Proficiency"] = hist_top10["US History Proficiency"].astype("float64")
hist_top10["Science Proficiency"] = hist_top10["Science Proficiency"].astype("float64")
hist_top10_copy = hist_top10[["School_Year", "District_Name", "Subgroup", "Math Proficiency", "English Proficiency", "US History Proficiency", "Science Proficiency"]].copy()
alt.Chart(hist_top10_copy).mark_line().encode(
    x = "School_Year", 
    y = "English Proficiency",
    color = 'Subgroup',
    facet = "District_Name"
)

alt.Chart(...)

# 2024-2025 Subject Proficiencies

In [166]:
top_10_2024 = hist_top10_copy[hist_top10_copy["School_Year"] == "2024"]
disabilities = top_10_2024[top_10_2024["Subgroup"] == "Students with Disabilities"]
disabilities

,School_Year,District_Name,Subgroup,Math Proficiency,English Proficiency,US History Proficiency,Science Proficiency
87279,2024,West Point Consolidated School District,Students with Disabilities,14.3,10.7,20.8,20.6
87498,2024,Clarksdale Municipal School District,Students with Disabilities,10.7,10.8,16.7,27.5
87767,2024,Hazlehurst City School District,Students with Disabilities,6.2,5.9,8.3,13.2
98486,2024,Noxubee County School District,Students with Disabilities,15.2,10.0,NaN,21.2
99526,2024,South Pike School District,Students with Disabilities,20.7,21.3,NaN,25.0
103348,2024,Western Line School District,Students with Disabilities,28.8,21.1,NaN,30.8
103619,2024,Wayne County School District,Students with Disabilities,18.6,15.0,25.0,21.8
103855,2024,Wilkinson County School District,Students with Disabilities,3.7,1.8,16.7,7.1
104047,2024,Coffeeville School District,Students with Disabilities,30.6,30.8,NaN,37.5
104252,2024,Yazoo City Municipal School District,Students with Disabilities,12.2,11.1,NaN,13.3


Wilkinson County School District has as little as 1.8% of students with disabilities that are proficient at English. Approximately 20% of Mississippi's students with disabilities are proficient at English, in comparison.  

The school districts with the largest discrepancies between students with disabilities' graduation rates and students without graduation rates' have bleak testing scores. Approximately 20% of students with disabilities in these school districts are proficient in english, math, history and science. 

In [167]:
alt.Chart(top_10_2024).mark_bar().encode(
    x = "District_Name",
    y = "Math Proficiency",
    xOffset="Subgroup:N",
    color="Subgroup:N"
)

alt.Chart(...)